# 01 · Data Ingestion

**Urban Flood Intelligence Platform** — Step 1 of the operational pipeline.

This notebook ingests the raw geospatial inputs for the study area (Nairobi, Kenya):

- **DEM** (SRTM 30 m) — download real data first, or fall back to synthesis.
- **Rainfall field** (CHIRPS-style 24h accumulation).
- **Rainfall time-series** (bimodal wet-season regime).
- **Drainage network** & **historical flood points** (vector layers).

### Real SRTM download (recommended)

```bash
python scripts/download_dem.py
```

Source: [OpenTopography SRTM GL1 COG VRT](https://opentopography.s3.sdsc.edu/raster/SRTM_GL1/SRTM_GL1_srtm.vrt) — streams only the Nairobi clip (~30 m resolution).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src import data_loader, utils
from src.utils import STUDY_AREA, PROCESSED_DIR

utils.ensure_dirs()
print('Study area :', STUDY_AREA.name)
print('Bounds     :', STUDY_AREA.bounds)
print('Grid       :', f'{STUDY_AREA.grid_size} x {STUDY_AREA.grid_size} (~{STUDY_AREA.cell_size_m:.0f} m/cell)')

## Ingest all raw layers

`ingest_all` returns the DEM grid, rainfall field, rainfall time-series, drainage network and flood-point layer in a single call.

In [ ]:
data = data_loader.ingest_all(base_rainfall_mm=utils.RAINFALL_REFERENCE_MM)
grid = data['grid']
rainfall_field = data['rainfall_field']
ts = data['rainfall_timeseries']
drainage = data['drainage']
floods = data['flood_points']

print('DEM shape        :', grid.shape)
print('Elevation range  :', f'{grid.elevation.min():.0f} - {grid.elevation.max():.0f} m')
print('Rainfall mean    :', f'{rainfall_field.mean():.1f} mm')
print('Drainage lines   :', len(drainage))
print('Flood incidents  :', len(floods))

## Quick-look: DEM & rainfall field

In [ ]:
extent = [STUDY_AREA.min_lon, STUDY_AREA.max_lon, STUDY_AREA.min_lat, STUDY_AREA.max_lat]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
im0 = ax[0].imshow(grid.elevation, cmap='terrain', extent=extent)
ax[0].set_title('Digital Elevation Model'); fig.colorbar(im0, ax=ax[0], shrink=0.8, label='m')
im1 = ax[1].imshow(rainfall_field, cmap='YlGnBu', extent=extent)
ax[1].set_title('Rainfall field (mm/24h)'); fig.colorbar(im1, ax=ax[1], shrink=0.8, label='mm')
plt.tight_layout(); plt.show()

In [ ]:
ax = ts.plot(x='date', y='rainfall_mm', figsize=(12, 3.5), legend=False,
             title='Daily rainfall (proxy CHIRPS) — bimodal wet seasons')
ax.set_ylabel('mm'); plt.tight_layout(); plt.show()

## Persist clean artefacts

In [ ]:
np.save(PROCESSED_DIR / 'dem.npy', grid.elevation)
np.save(PROCESSED_DIR / 'rainfall_field.npy', rainfall_field)
ts.to_csv(PROCESSED_DIR / 'rainfall_timeseries.csv', index=False)
floods.to_csv(PROCESSED_DIR / 'flood_points.csv', index=False)
print('Saved raw ingestion artefacts to', PROCESSED_DIR)
print('Next: 02_terrain_analysis.ipynb')